# LeetCode 27 — Remove Element | Nhập môn Python, giảng từng bước

**Độ khó:** Easy · **Kỹ thuật trọng tâm:** Two Pointers · in-place filtering  
**Bài gốc:** https://leetcode.com/problems/remove-element/

Notebook này dành cho người **mới bắt đầu**. Đọc các ô từ trên xuống: trước hết hiểu **vì sao**, sau đó chạy code nhỏ để tự thấy **cách làm**.

**Khung hàm (signature) cần nộp LeetCode:**

```python
class Solution:
    def removeElement(self, nums: list[int], val: int) -> int:
        ...
```

**Sau bài học, bạn có thể:**
- Hiểu đúng hợp đồng in-place: trả k và chỉnh phần đầu nums.
- Tự phát hiện tiêu chí đọc/giữ/ghi, áp dụng hai con trỏ.
- So sánh cách giữ thứ tự với cách đổi phần tử cuối (không cần giữ thứ tự).

**Cách sử dụng:** chạy từng code cell theo thứ tự bằng `Shift + Enter`. Những ô có `TODO` là bài tự luyện: chỉ định nghĩa hàm hoặc chứa lời nhắc kiểm thử, **không được gọi tự động** khi chưa làm xong.

> Lộ trình: hiểu đề → Python căn bản → cách trực giác → phát hiện nút thắt → cải tiến → trace → nộp LeetCode → test → lỗi → vận dụng → bài tập → đáp án → checklist.

## 1. Đề bài — bỏ tất cả phần tử bằng `val`

Cho `nums` và số `val`. Hãy loại bỏ **mọi phần tử bằng `val`** ngay trong mảng, rồi trả về **số phần tử không bằng `val`** gọi là `k`.

Với `nums=[3,2,2,3], val=3` → trả `k=2`, phần đầu là `[2,2]`.

Với `nums=[0,1,2,2,3,0,4,2], val=2` → `k=5`, các số đầu có thể là `[0,1,3,0,4]`. **Thứ tự các phần tử không bắt buộc phải giữ nguyên** (nhưng cách chính của chúng ta sẽ giữ để dễ hiểu).

Giống bài #26: **chỉ `nums[:k]` được chấm**, giá trị ở phần đuôi không quan trọng; không cần rút ngắn chiều dài list. **Khác #26:** mảng **không cần sort**, và tiêu chí loại bỏ dựa vào `val`, không dựa vào phần tử trùng.

In [ ]:
a = [3, 2, 2, 3]
val = 3
print("Input:", a, "| val =", val)
print("Số lượng cần giữ theo suy luận:", 2)

## 2. Python căn bản — duyệt các giá trị

`for x in nums:` lần lượt đọc **giá trị** của các phần tử, không phải index. `if x != val:` nghĩa là “nếu x **khác** val”.

`!=` là toán tử **không bằng**; `==` là **bằng**. `=` là phép **gán biến**, không được dùng thay `==` trong `if`.

In [ ]:
nums = [3, 2, 2, 3]
val = 3
for x in nums:
    print("x =", x, "| có giữ không?", x != val)

## 3. Cách trực giác — tạo list mới

Dễ nghĩ nhất: lọc sang list khác bằng **list comprehension**: `[x for x in nums if x != val]`. Cú pháp này có nghĩa là “lấy x từ nums nếu x khác val”.

Nhưng cách này cấp phát list mới **O(n) bộ nhớ**, trong khi đề yêu cầu sửa chính `nums` **tại chỗ**, với O(1) bộ nhớ phụ. Vì vậy đây là cách **tham khảo** để kiểm tra kết quả, không phải code submit.

In [ ]:
def remove_element_new_list(nums, val):
    return [x for x in nums if x != val]

print(remove_element_new_list([0,1,2,2,3,0,4,2], 2))

## 4. Cách xóa trực tiếp nhưng chậm

Cách khác là `.pop(i)` khi gặp giá trị cần xóa. Nhưng mỗi lần `pop` ở giữa list phải dời các phần tử phía sau, nên trong trường hợp xấu nhất tốn **O(n²)** thời gian. Hơn nữa cách này làm **độ dài list thay đổi**, không cần thiết cho cơ chế chấm LeetCode.

Ta nên **ghi đè tại phần đầu** thay vì xóa và dồn nhiều lần.

In [ ]:
demo = [3, 2, 2, 3]
print("Trước pop:", demo)
demo.pop(0)
print("Sau pop(0):", demo, "| chiều dài thay đổi:", len(demo))

## 5. Insight — chỉ cần giữ phần tử KHÁC val

Dùng hai biến:

- `read`: đang **đọc** phần tử nào (hoặc đọc trực tiếp bằng `for x in nums`).
- `write = k`: vị trí **ghi** giá trị tiếp theo cần giữ.

Nếu gặp `x == val`, **bỏ qua**. Nếu gặp `x != val`, ghi `nums[write] = x`, rồi tăng `write`.

Trực giác: hai đầu list — một người đọc lướt qua từng phần tử; một người xếp các phần tử hợp lệ thành dãy liên tục ở đầu list.

## 6. Bảng trace: `[0,1,2,2,3]`, val=2

| read | x | write trước | giữ? | prefix hợp lệ sau bước |
|---:|---:|---:|---|---|
| 0 | 0 | 0 | có → ghi nums[0] | `[0]` |
| 1 | 1 | 1 | có → ghi nums[1] | `[0,1]` |
| 2 | 2 | 2 | không | `[0,1]` |
| 3 | 2 | 2 | không | `[0,1]` |
| 4 | 3 | 2 | có → ghi nums[2] | `[0,1,3]` |

Kết thúc `write=3`, trả 3 và `nums[:3]=[0,1,3]`. Không cần biết `nums[3:]` còn gì.

## 7. Code chính — ghi đè vào phần đầu

Giữ nguyên thứ tự phần tử hợp lệ. Chỉ có một lần duyệt, không tạo list mới, và `write` không vượt quá vị trí đang đọc. Vì vậy ghi đè **không làm mất phần tử chưa xử lý**.

In [ ]:
def remove_element_inplace(nums: list[int], val: int) -> int:
    write = 0
    for x in nums:
        if x != val:
            nums[write] = x
            write += 1
    return write

a = [3, 2, 2, 3]
k = remove_element_inplace(a, 3)
print("k =", k, "| nums[:k] =", a[:k], "| cả mảng =", a)

## 8. Tại sao vòng `for x in nums` vẫn an toàn khi đang ghi nums?

Dù ta đang sửa các phần tử trong list, biến `write` **không bao giờ lớn hơn** vị trí phần tử đang đọc. Chỉ những vị trí đã đọc mới có thể bị ghi đè, còn các vị trí chưa đọc giữ nguyên.

Ví dụ `read=4`, `write=2`: ghi `nums[2] = nums[4]` là an toàn vì vị trí 2 đã duyệt qua. Nếu `write > read`, ta có thể phá hỏng dữ liệu tương lai — nhưng thuật toán này không cho phép xảy ra.

**Bất biến:** sau mỗi bước, `nums[:write]` chứa đúng các phần tử **khác `val` đã đọc**, theo thứ tự ban đầu.

## 9. Trace chạy thật với log trước/sau

Trace minh họa sẽ tạo `.copy()` chỉ để **không sửa input người học**; lời giải nộp không có copy. Mỗi dòng sẽ cho biết quyết định giữ hay bỏ, và phần đầu mảng lúc đó.

In [ ]:
def trace_remove_element(nums, val):
    nums = nums.copy()   # Chỉ phục vụ giảng dạy
    write = 0
    print("Bắt đầu:", nums, "| cần bỏ:", val)
    for read in range(len(nums)):
        x = nums[read]
        print(f"read={read}, x={x}, write={write}", end=" ")
        if x != val:
            nums[write] = x
            write += 1
            print("-> GIỮ; prefix=", nums[:write])
        else:
            print("-> BỎ QUA; prefix=", nums[:write])
    print("KẾT THÚC: k=", write, "| mảng:", nums)
    return write

assert trace_remove_element([0,1,2,2,3,0,4,2], 2) == 5

## 10. Code LeetCode chuẩn

LeetCode cần phương thức tên `removeElement(self, nums, val)` và giá trị trả về là **int k**. Không được trả `nums` hay `nums[:k]`.

In [ ]:
class Solution:
    def removeElement(self, nums: list[int], val: int) -> int:
        k = 0
        for x in nums:
            if x != val:
                nums[k] = x
                k += 1
        return k

nums = [0,1,2,2,3,0,4,2]
k = Solution().removeElement(nums, 2)
print("return k =", k, "| prefix:", nums[:k])

## 11. Trường hợp mảng rỗng, tất cả bị xóa, không có số cần xóa

Không phải lúc nào cũng có số cần xóa. Ba tình huống nên test:

- `nums=[]` ⇒ k=0, không có ô cần đọc.
- `nums=[2,2,2], val=2` ⇒ k=0, phần đầu hợp lệ rỗng.
- `nums=[1,3,5], val=2` ⇒ k=3, giữ nguyên tất cả.

Nếu tất cả phần tử bị xóa, **không cần** biến `nums` thành `[]`. Chỉ cần `k=0`.

In [ ]:
for nums, val in [([], 2), ([2,2,2], 2), ([1,3,5], 2)]:
    data = nums.copy()
    k = Solution().removeElement(data, val)
    print(f"nums={nums}, val={val} -> k={k}, prefix={data[:k]}")

## 12. Tối ưu khác — đổi với phần tử cuối (không giữ thứ tự)

Đề **không yêu cầu** thứ tự nên cũng có thể đổi một phần tử bằng `val` với phần tử **cuối vùng chưa xử lý**, rồi thu ngắn vùng ấy. Cách này có thể giảm số lần ghi khi phần tử cần xóa ít nhưng có thể **đổi thứ tự**.

- `i` đang xét; `n` là độ dài vùng hợp lệ (chưa loại).
- Nếu `nums[i] == val`, gán `nums[i] = nums[n-1]`, rồi giảm `n`. **Không tăng `i`** vì phần tử mới đổi đến vẫn chưa được xét.
- Nếu khác `val`, tăng `i`.

Trả `n`; chỉ `nums[:n]` quan trọng. Cả hai cách đều **O(n)** thời gian, **O(1)** bộ nhớ.

In [ ]:
def remove_element_swap_end(nums: list[int], val: int) -> int:
    i, n = 0, len(nums)
    while i < n:
        if nums[i] == val:
            nums[i] = nums[n - 1]
            n -= 1
        else:
            i += 1
    return n

a = [0, 1, 2, 2, 3, 0, 4, 2]
k = remove_element_swap_end(a, 2)
print("Kết quả có thể khác thứ tự:", a[:k], "| k:", k)

## 13. Vì sao hai cách đều đúng?

**Cách ghi đè:** sau khi đọc một đoạn đầu, vùng `nums[:write]` chứa chính xác mọi phần tử không bằng val đã đọc. Cuối vòng lặp, `write` bằng số phần tử cần giữ nên hợp lệ.

**Cách đổi cuối:** vùng `nums[:n]` giữ những phần tử **chưa bị loại**; mỗi lần gặp `val`, ta chuyển phần tử cuối chưa xét vào vị trí đó và giảm kích thước vùng. Không tăng `i` vì phải kiểm tra phần tử vừa chuyển đến. Hết vòng, mọi phần tử trong `nums[:n]` khác val.

Gọi `n0` là độ dài ban đầu. Cả hai cách đều O(n0) thời gian, O(1) bộ nhớ; cách swap có thể đảo thứ tự nên chỉ dùng khi đề cho phép.

## 14. Kiểm thử cách chính và cách swap

Với cách swap, thứ tự không cố định, nên chỉ kiểm tra **multiset** (giá trị và số lần xuất hiện) bằng `sorted(prefix)`; còn cách ghi đè thì so đúng thứ tự với list mới lọc được.

In [ ]:
tests_27 = [
    ([3,2,2,3], 3, [2,2]),
    ([0,1,2,2,3,0,4,2], 2, [0,1,3,0,4]),
    ([], 0, []), ([2,2,2], 2, []), ([1,3,5], 2, [1,3,5]),
    ([-1,0,-1,2], -1, [0,2]), ([1], 1, []),
]
for original, val, expected in tests_27:
    nums = original.copy(); length = len(nums)
    k = Solution().removeElement(nums, val)
    assert k == len(expected) and nums[:k] == expected
    assert len(nums) == length
    nums2 = original.copy()
    k2 = remove_element_swap_end(nums2, val)
    assert k2 == len(expected) and sorted(nums2[:k2]) == sorted(expected)
print("PASS", len(tests_27), "ca + so sánh 2 chiến lược")

## 15. Sai lầm thường gặp

- **Trả list thay int:** `return nums[:k]` sai format; phải `return k`.
- **`nums = result` không sửa list input:** biến `nums` bên trong chỉ trỏ tới list mới, caller vẫn giữ object cũ.
- **Dùng `.remove(val)` một lần:** chỉ xóa **một** lần xuất hiện, không phải tất cả.
- **Đổi phần tử cuối nhưng vẫn `i += 1` ngay:** có thể bỏ sót phần tử `val` mới được đưa từ cuối lên.
- **Quên xét list rỗng và tất cả bằng val:** code chính tự xử lý được với `k=0`.

In [ ]:
a = [2, 2, 3, 2]
b = a.copy()
b.remove(2)  # xóa một lần thôi
print("Xóa một lần:", b)
print("Giữ đúng các số khác 2:", remove_element_new_list(a, 2))

## 16. Pattern: lọc tại chỗ

Khi bài yêu cầu **giữ lại một số phần tử, không cấp phát array mới**, dùng `write` để xây **prefix đúng**; `read` sẽ đi qua toàn bộ dữ liệu. Với bài #26, tiêu chí giữ là **giá trị mới so với phần tử duy nhất cuối**. Với bài #27, tiêu chí giữ là **khác val**.

**Mẹo chuyển giao:** Có thể thay `if x != val:` bằng bất cứ điều kiện giữ nào, ví dụ giữ số chẵn, giữ số dương, giữ số không phải 0 — miễn là hiểu rõ đề có cho phép đổi thứ tự hay không.

## 17. Bài tập tự luyện (5 bài) — chưa gọi hàm TODO

### Bài 1 — Hiểu điều kiện khác val — Tự làm

Viết `is_kept(x,val)` trả True nếu x **khác** val.

**Gợi ý cấp 1 (tư duy):** Chỉ giữ khi khác.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Toán tử `!=`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`return x != val`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_27_1(x: int, val: int) -> bool:
    # TODO: so sánh
    pass

# assert exercise_27_1(2, 3) is True
# assert exercise_27_1(3, 3) is False

### Bài 2 — Tính bằng tay — Tự làm

Cho `[4,4,1,2,4,3]`, `val=4`: số `k` bằng bao nhiêu?

**Gợi ý cấp 1 (tư duy):** Gạch tất cả số 4.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Đếm số còn lại.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Còn 1,2,3.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_27_2() -> int:
    # TODO: đếm số khác 4
    pass

# assert exercise_27_2() == 3

### Bài 3 — Viết lại read/write — Tự làm

Hoàn thiện thuật toán ghi đè trong chính `nums`, trả `k`.

**Gợi ý cấp 1 (tư duy):** Chỉ các phần tử được giữ mới làm tăng k.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`for x in nums`, `nums[k] = x`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Sau khi ghi x, đừng quên `k += 1`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_27_3(nums: list[int], val: int) -> int:
    # TODO: k=0, for x in nums, if x!=val thì ghi nums[k]=x rồi tăng k
    pass

# a=[3,2,2,3]
# assert exercise_27_3(a,3)==2 and a[:2]==[2,2]

### Bài 4 — Đổi cuối không giữ thứ tự — Tự làm

Tự viết cách đổi phần tử cuối và nhớ **không tăng i** sau khi gặp val.

**Gợi ý cấp 1 (tư duy):** Vùng chưa xét là nums[i:n].

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

Dùng `while i < n`.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

`if nums[i]==val: nums[i]=nums[n-1]; n-=1; else: i+=1`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_27_4(nums: list[int], val: int) -> int:
    # TODO: i=0, n=len(nums)
    # TODO: while i<n: nếu trùng thì đổi với nums[n-1], giảm n;
    # TODO: nếu khác thì tăng i
    pass

# a=[3,2,2,3]
# k=exercise_27_4(a,3)
# assert k==2 and sorted(a[:k])==[2,2]

### Bài 5 — Mini challenge giữ số chẵn — Tự làm

Biến thể: viết `keep_even(nums)` ghi **mọi số chẵn** vào đầu mảng theo thứ tự cũ, trả k.

**Gợi ý cấp 1 (tư duy):** Điều kiện giữ bây giờ không còn phụ thuộc val.

<details><summary>Gợi ý cấp 2 — Cú pháp</summary>

`x % 2 == 0` nghĩa là x chia hết cho 2.

</details>

<details><summary>Gợi ý cấp 3 — Gần code</summary>

Dùng cùng code bài 27 và thay `if x != val` bằng `if x % 2 == 0`.

</details>

**Khi làm xong**, bỏ dấu `#` ở lệnh `assert` gợi ý trong ô code dưới để tự kiểm tra. Đáp án đầy đủ nằm ở phần cuối notebook.

In [ ]:
def exercise_27_5(nums: list[int]) -> int:
    # TODO: nếu x%2==0 thì giữ bằng read/write
    pass

# a=[1,2,3,4,6]
# assert exercise_27_5(a)==3 and a[:3]==[2,4,6]

## 18. Đáp án tham khảo — xem cuối cùng

In [ ]:
def answer_27_1(x, val):
    return x != val

def answer_27_2():
    return 3

def answer_27_3(nums, val):
    k = 0
    for x in nums:
        if x != val:
            nums[k] = x
            k += 1
    return k

def answer_27_4(nums, val):
    i = 0; n = len(nums)
    while i < n:
        if nums[i] == val:
            nums[i] = nums[n - 1]
            n -= 1
        else:
            i += 1
    return n

def answer_27_5(nums):
    k = 0
    for x in nums:
        if x % 2 == 0:
            nums[k] = x
            k += 1
    return k

assert answer_27_1(2,3) and not answer_27_1(3,3)
assert answer_27_2() == 3
u=[3,2,2,3]; assert answer_27_3(u,3)==2 and u[:2]==[2,2]
u=[3,2,2,3]; assert answer_27_4(u,3)==2 and sorted(u[:2])==[2,2]
u=[1,2,3,4,6]; assert answer_27_5(u)==3 and u[:3]==[2,4,6]
print("Đáp án tham khảo: PASS 5/5")

## Tóm tắt — mẫu tư duy cần nhớ

**Mẫu:** `k=0`; đọc từng `x` trong `nums`; nếu `x != val`: `nums[k]=x; k+=1`; return `k`. **O(n)** thời gian, **O(1)** bộ nhớ phụ. **Mở rộng:** đổi với cuối khi đề không cần thứ tự.

## Checklist trước khi submit

- [ ] Tôi hiểu vì sao cần trả số nguyên k.
- [ ] Tôi chỉ xem nums[:k] là phần được chấm.
- [ ] Tôi không nhầm xóa khỏi list với ghi tại chỗ.
- [ ] Tôi hiểu sự khác biệt #26 (sorted / unique) và #27 (lọc !=val).
- [ ] Tôi đã test mảng rỗng, chỉ gồm val, không chứa val.

**Tự kiểm tra lần cuối:** có thể giải thích lời giải bằng lời trước khi xem code? Có tự tính được ít nhất một bảng trace? Nếu chưa, quay lại phần mô phỏng.

*Notebook học tập được biên soạn theo cẩm nang tiếng Việt đính kèm; tiêu đề/đề/ràng buộc bám sát LeetCode. Những biến thể được đánh dấu là mở rộng, không phải yêu cầu đề chính.*